In [ ]:
from typing import List, TypedDict, Annotated
import time
import operator
import re
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

# FIX 1: HuggingFaceEndpoint & ChatHuggingFace import added
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()


ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
docs = PyPDFLoader("./documents/book1.pdf").load()

In [ ]:
docs = PyPDF

In [ ]:
split_docs = RecursiveCharacterTextSplitter(chunk_size = 900,chunk_overlap= 100).split_documents(docs)

for d in split_docs:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [ ]:

print(len(split_docs))

2433


In [ ]:
embed_model = HuggingFaceEmbeddings(model= "sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4582.24it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
vector_store = FAISS.from_documents(split_docs,embed_model)

In [31]:
retriever = vector_store.as_retriever(search_type = "similarity", search_kwargs = {'k': 4})


In [ ]:

class State(TypedDict):
    question : str
    docs : list[Document]

    strips : list[str]
    kept_strips: list[str]
    refined_context: str
    answer : str

In [20]:

from dotenv import load_dotenv

load_dotenv()  # ✅ এটা আগে call করতে হবে

# Debug: দেখো key আসছে কিনা
print(os.getenv("GROQ_API_KEY"))  # None হলে সমস্যা আছে

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)


***


In [32]:
def retrieve(state):
    q = state['question']
    return {'docs': retriever.invoke(q)}

In [ ]:
# retriever sentence striper funtion
def decompose_to_sentence(text: str) -> list[str]:
    text = re.sub(r"\s+", " ", text).strip()

    # Step 1: sentence boundary দিয়ে আগে ভাগ করো
    raw_sentences = re.split(r"(?<=[.!?])\s+", text)

    # Step 2: ছোট sentence গুলোকে জোড়া লাগিয়ে 200 char-এর chunk বানাও
    chunks = []
    current_chunk = ""

    for s in raw_sentences:
        s = s.strip()
        if not s:
            continue
        # যদি current_chunk-এ s যোগ করলে 200 পার হয়
        if len(current_chunk) + len(s) + 1 > 400 and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = s  # নতুন chunk শুরু হবে এই sentence দিয়ে
        else:
            current_chunk = (current_chunk + " " + s).strip()

    # শেষে যা বাকি থাকবে সেটাও রাখো
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks


In [ ]:
# Filter



filter_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", 
         "You are a strict relevance filter. \n"
         "Does the sentence directly help answer the question? \n"
         "Reply with ONLY one word: YES or NO. Nothing else."
        ),
        ("human", "Question: {question}\n\nSentence:{sentence}")
    ]
) 

filter_chain = filter_prompt | llm | StrOutputParser() | (lambda x: "YES" in x.strip().upper())



NameError: name 'ChatPromptTemplate' is not defined

In [ ]:
def refine(state: State) -> State:
    q = state["question"]
    docs = state['docs']

    # সব doc-এর sentence একসাথে prepare করো
    doc_strips = []
    all_inputs = []

    for doc in docs:
        strips = decompose_to_sentence(doc.page_content)
        doc_strips.append(strips)
        for s in strips:
            all_inputs.append({'question': q, 'sentence': s})

    # একটাই batch call
    all_results = filter_chain.batch(
        all_inputs,
        config={"max_concurrency": 10}
    )

    # results ভাগ করো প্রতিটি doc-এ
    all_strips = []       
    all_kept_strips = []
    idx = 0

    for strips in doc_strips:
        count = len(strips)
        current_r = all_results[idx: idx + count]
        idx += count

        all_strips.extend(strips)  
        kept = []
        for s,keep in zip(strips,current_r):
            if keep:
                kept.append(s)

        all_kept_strips.extend(kept)

    # Final context তৈরি
    final_context = "\n".join(all_kept_strips).strip()
    

    return {
        "kept_strips": all_kept_strips,
        "refined_context": final_context,
        "strips": all_strips,
        }


In [25]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ('system',"Answer only from the context. If not in contex, say you don't know",),
        ('human', "Question : {question}\n\nContext:\n{context}")
    ]
)


def generate(state: State) -> State:
    generate_chain = answer_prompt | llm | StrOutputParser()  # ✅ যোগ করো
    out = generate_chain.invoke({
        "question": state["question"], 
        "context": state['refined_context']
    })
    return {"answer": out}  # ✅ out এখন string, .content না



In [26]:
# FIX 4: Removed duplicate StateGraph definition
g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("refine", refine)
g.add_node("generate", generate)

g.add_edge(START, "retrieve")
g.add_edge("retrieve", "refine")
g.add_edge("refine", "generate")
g.add_edge("generate", END)

app = g.compile()
print("Graph compiled successfully!")


Graph compiled successfully!


In [40]:
# FIX 5: Typo fixed ('whte' -> 'What') + all State keys provided
res = app.invoke({
    "question": "What is Neural Networks?",
    "docs": [],
    "r": [],
    "strips": [],
    "kept_strips": [],
    "refined_context": "",
    "keep_refined_chunks": [],
    "answer": ""
})

print("Answer:", res["answer"])
print("Filter results count:", len(res['r']))
print("Kept sentences count:", len(res['kept_strips']))


Answer: In the context of the problem, Neural Networks refer to a type of machine learning model that attempts to mimic the structure and function of the human brain. It is also known as the multilayer perceptron (MLP) due to its ability to process information in multiple layers.

To learn the parameters w of a neural network from data using maximum likelihood, we can follow these steps:

1. Evaluate the derivatives of the error function with respect to the weights (w) at each step. This is typically done using the chain rule of calculus.
2. The error function is minimized using an iterative procedure, with adjustments to the weights (w) being made in a sequence of steps.
3. The maximum likelihood approach involves finding the parameters (w) that maximize the likelihood of observing the given data.

In the context of a neural network, the maximum likelihood approach can be implemented using algorithms such as stochastic gradient descent (SGD), which iteratively updates the weights (w) 

In [38]:
print(res['keep_refined_chunks'])
print(len(res['keep_refined_chunks']))

[{'chunk_no': 0, 'chunk': 'multiple perceptrons (with discontinuous nonlinearities). For many applications, the\nresulting model can be signiﬁcantly more compact, and hence faster to evaluate, than\na support vector machine having the same generalization performance. The price to\nbe paid for this compactness, as with the relevance vector machine, is that the like-\nlihood function, which forms the basis for network training, is no longer a convex\nfunction of the model parameters. In practice, however, it is often worth investing\nsubstantial computational resources during the training phase in order to obtain a\ncompact model that is fast at processing new data.\nThe term ‘neural network’ has its origins in attempts to ﬁnd mathematical rep-\nresentations of information processing in biological systems (McCulloch and Pitts,\n1943; Widrow and Hoff, 1960; Rosenblatt, 1962; Rumelhart et al., 1986). Indeed,', 'r': [False, False, True], 'strips': ['multiple perceptrons (with discontinuous 

In [ ]:
print(res["docs"][0].page_content)
print('*'*100)
print(res["docs"][1].page_content)
print('*'*100)
print(res["docs"][2].page_content) 
print('*'*100)
print(res["docs"][3].page_content)
print('*'*100)

multiple perceptrons (with discontinuous nonlinearities). For many applications, the
resulting model can be signiﬁcantly more compact, and hence faster to evaluate, than
a support vector machine having the same generalization performance. The price to
be paid for this compactness, as with the relevance vector machine, is that the like-
lihood function, which forms the basis for network training, is no longer a convex
function of the model parameters. In practice, however, it is often worth investing
substantial computational resources during the training phase in order to obtain a
compact model that is fast at processing new data.
The term ‘neural network’ has its origins in attempts to ﬁnd mathematical rep-
resentations of information processing in biological systems (McCulloch and Pitts,
1943; Widrow and Hoff, 1960; Rosenblatt, 1962; Rumelhart et al., 1986). Indeed,
****************************************************************************************************
for this reason th